# Notebook 34 — Is the CNN collapse an input-interface artefact of layer-wise pruning?

Notebooks 32 and 33 rejected the capacity hypothesis: at matched or lower remaining capacity the MLP
(co-equal arm) loses 0.013 macro-F1 with zero classes affected and the transformer degrades gracefully,
while the CNN loses 0.271 with 17 classes affected. Before attributing this to the convolutional locality
prior, one structural fact must be tested: the CNN's first layer `conv.0` (Conv1d 1->64, kernel 3) has only
192 weights, and the archived pruner is layer-wise uniform, so prune80 leaves 38 taps for the entire input
interface. The MLP's input layer keeps ~1,000 weights at prune90; the transformer's tokenizer was protected.

**Hypothesis H-input (stated before running):** the CNN collapse is input-interface starvation caused by
layer-wise pruning of a tiny first layer, not a property of convolution per se.

**Design.** On the five independently trained CNN baselines from Notebook 11 (`M0_paired`, seeds 0-4),
three prune80 policies: (A) archived layer-wise uniform 80% on every layer (reference, loaded from the saved
`prune80_paired` checkpoints); (B) layer-wise 80% with `conv.0` protected (192 weights untouched, 0.7% of
prunable weights); (C) global magnitude 80% across all prunable weights (a single threshold; small
high-magnitude layers keep more). Same fine-tuning protocol. Per-layer resulting sparsities are recorded.

**Gate.** H-input is supported if policy B eliminates the collapse (mean macro-F1 loss < 0.10 and fewer
than four classes materially affected in >=3/5 seeds) while reference A reproduces it. Probes and a
dense-head refit are run on B and C at the anchor seed. Resumable per (seed, policy). GPU runtime required.

In [ ]:
# --- Colab bootstrap ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys, copy, json as _json
os.chdir(REPO); sys.path.insert(0, REPO)
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.utils.prune as prune
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from src.config import CFG, PATHS, set_all_seeds
from src.data import load_raw, clean, temporal_within_capture_split
from src import train as TR, models as M, explain as EXP, mitigate
from src.comnet_audit import assign_validation_tiers, calibration_summary, environment_record, write_json
from src.train import load_anchor, predict, per_class_recall_table, feature_columns

assert torch.cuda.is_available(), 'switch to a GPU runtime first'
DEVICE = TR.DEVICE
DATASET, ARCH = 'ciciot2023', 'cnn1d'
SEEDS = list(CFG['seeds']); ANCHOR = int(CFG['anchor_seed'])
OUT = PATHS.tables('comnet')
PRACTICAL_LOSS = 0.10
POLICIES = ['layerwise80_all', 'layerwise80_protect_conv0', 'global80']

ARCH_KW = {'channels': (64, 128)}
print('ARCH_KW:', ARCH_KW, '| policies:', POLICIES)

In [ ]:
df = clean(load_raw(DATASET, subsample=True, seed=ANCHOR), DATASET)
splits = temporal_within_capture_split(df, seed=ANCHOR)
feat_cols = feature_columns(df)
print(f'{len(df):,} rows | {df.label.nunique()} classes')

In [ ]:
# Pruning policies on the CNN. Fine-tune loop identical to src.compression.prune_and_finetune.
def prunable(model):
    return [(mod, 'weight') for mod in model.modules() if isinstance(mod, (nn.Linear, nn.Conv1d))]

def layer_names(model):
    return {mod: n for n, mod in model.named_modules()}

def apply_policy(model, policy, amount=0.80):
    m = copy.deepcopy(model); names = layer_names(m)
    if policy == 'layerwise80_all':
        for mod, name in prunable(m): prune.l1_unstructured(mod, name=name, amount=amount); prune.remove(mod, name)
    elif policy == 'layerwise80_protect_conv0':
        for mod, name in prunable(m):
            if names[mod] == 'conv.0': continue
            prune.l1_unstructured(mod, name=name, amount=amount); prune.remove(mod, name)
    elif policy == 'global80':
        params = prunable(m)
        prune.global_unstructured(params, pruning_method=prune.L1Unstructured, amount=amount)
        for mod, name in params: prune.remove(mod, name)
    else:
        raise ValueError(policy)
    return m

def layer_sparsity(model):
    names = layer_names(model); out = {}
    for mod, name in prunable(model):
        w = getattr(mod, name); out[names[mod]] = float((w == 0).float().mean())
    z = sum(int((getattr(mod, n) == 0).sum()) for mod, n in prunable(model)); n_ = sum(getattr(mod, n).numel() for mod, n in prunable(model))
    out['prunable_sparsity'] = z / n_; out['remaining_nonzero_prunable'] = n_ - z
    return out

def finetune_masked(model, seed, *, ft_epochs=8, batch_size=4096, lr=5e-4, verbose=False):
    set_all_seeds(seed)
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    le = LabelEncoder().fit(df['label'].to_numpy())
    scaler = StandardScaler().fit(df.loc[splits['train'], feat_cols].to_numpy(np.float32))
    t = TR.make_tensors(df, splits, feat_cols, le, scaler); Xtr, ytr = t['train']
    model = model.to(DEVICE)
    masks = {(mod, name): (getattr(mod, name) != 0).float() for mod, name in prunable(model)}
    hooks = [getattr(mod, name).register_hook((lambda mk: (lambda g: g * mk))(mk)) for (mod, name), mk in masks.items()]
    w = TR.tempered_class_weights(ytr.numpy(), len(le.classes_))
    crit = nn.CrossEntropyLoss(weight=w); opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    for ep in range(ft_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
        if verbose: print(f'    ft epoch {ep}')
    for h in hooks: h.remove()
    with torch.no_grad():
        for mod, name in prunable(model): getattr(mod, name).mul_((getattr(mod, name) != 0).float())
    return model.eval(), le, scaler

def save_ckpt(model, le, scaler, path):
    torch.save({'state_dict': model.state_dict(), 'classes': list(le.classes_), 'feat_cols': feat_cols,
                'scaler_mean': scaler.mean_, 'scaler_scale': scaler.scale_}, path)

# structural fact check on the anchor-seed baseline
m0_chk, _, _, _ = load_anchor(DATASET, ARCH, 'M0_paired', ANCHOR, arch_kwargs=ARCH_KW)
for mod, n in prunable(m0_chk): print(f'  {layer_names(m0_chk)[mod]:8s} {getattr(mod, n).numel():>7,} weights -> layer-wise 80% keeps {int(getattr(mod, n).numel() * 0.2):,}')

In [ ]:
# Run the three policies on the five paired CNN baselines, with resume
baseline_val, baseline_test, comp_test, macro, ls_rows, models_out = {}, {}, {}, [], [], {}
for seed in SEEDS:
    print(f'\n===== seed {seed} =====')
    m0, le, scaler, _ = load_anchor(DATASET, ARCH, 'M0_paired', seed, arch_kwargs=ARCH_KW)
    yv, pv, _ = predict(m0, df, splits, le, scaler, feat_cols, which='val')
    yt, pt, _ = predict(m0, df, splits, le, scaler, feat_cols, which='test')
    baseline_val[seed] = per_class_recall_table(yv, pv, le).set_index('label')['recall']
    baseline_test[seed] = per_class_recall_table(yt, pt, le).set_index('label')['recall']
    macro.append({'seed': seed, 'cell': 'M0', 'test_macro_f1': f1_score(yt, pt, average='macro')})
    for policy in POLICIES:
        if policy == 'layerwise80_all':
            p_c = PATHS.model(DATASET, ARCH, 'prune80_paired', seed)      # archived Notebook 11 checkpoint
        else:
            p_c = PATHS.model(DATASET, ARCH, f'{policy}_paired', seed)
        if os.path.exists(p_c):
            mp = M.build(ARCH, len(feat_cols), len(le.classes_), **ARCH_KW).to(DEVICE)
            ck = torch.load(p_c, map_location=DEVICE, weights_only=False)
            mp.load_state_dict(ck['state_dict'] if isinstance(ck, dict) and 'state_dict' in ck else ck); mp.eval(); print(f'  loaded {policy}')
        else:
            assert policy != 'layerwise80_all', 'archived prune80_paired checkpoint missing; run Notebook 11 with SAVE_CHECKPOINTS=True'
            mp, _, _ = finetune_masked(apply_policy(m0, policy), seed, verbose=True); save_ckpt(mp, le, scaler, p_c); print(f'  saved {policy}')
        ls = layer_sparsity(mp); ls_rows.append({'seed': seed, 'cell': policy, **ls})
        yt, pc, _ = predict(mp, df, splits, le, scaler, feat_cols, which='test')
        comp_test[(seed, policy)] = per_class_recall_table(yt, pc, le).set_index('label')['recall']
        macro.append({'seed': seed, 'cell': policy, 'test_macro_f1': f1_score(yt, pc, average='macro')})
        models_out[(seed, policy)] = mp
print('\nall policies complete')

In [ ]:
# Aggregate (Notebook 11 schema)
tiers = assign_validation_tiers(pd.DataFrame(baseline_val))
rows = []
for (seed, cell), rc in comp_test.items():
    r0 = baseline_test[seed]
    for cls in r0.index.intersection(rc.index):
        loss = float(r0.loc[cls] - rc.loc[cls]); band = float(tiers.loc[cls, 'validation_2sd_band'])
        rows.append({'seed': seed, 'cell': cell, 'class': cls, 'M0_test_recall': float(r0.loc[cls]), 'compressed_test_recall': float(rc.loc[cls]),
                     'recall_loss': loss, 'validation_tier': tiers.loc[cls, 'validation_tier'], 'validation_2sd_band': band,
                     'crosses_validation_band': bool(loss > band), 'practically_material': bool(loss >= PRACTICAL_LOSS),
                     'material_and_beyond_band': bool((loss > band) and (loss >= PRACTICAL_LOSS))})
eff = pd.DataFrame(rows); assert len(eff) > 0, 'no rows: run the policy cell in this session first'
eff.to_csv(OUT / 'cnn_policy_per_class_effects.csv', index=False)
summ = eff.groupby(['cell', 'class']).agg(mean_recall_loss=('recall_loss', 'mean'), sd_recall_loss=('recall_loss', 'std'),
        affected_frequency=('material_and_beyond_band', 'mean'), n=('seed', 'nunique')).reset_index()
summ.to_csv(OUT / 'cnn_policy_per_class_summary.csv', index=False)
mdf = pd.DataFrame(macro); mdf.to_csv(OUT / 'cnn_policy_macro_f1_wide.csv', index=False)
msum = mdf.groupby('cell')['test_macro_f1'].agg(['count', 'mean', 'std', 'min', 'max']).reset_index()
msum.columns = ['cell', 'n', 'mean', 'sd', 'min', 'max']; msum.to_csv(OUT / 'cnn_policy_macro_f1_summary.csv', index=False)
lsdf = pd.DataFrame(ls_rows); lsdf.to_csv(OUT / 'cnn_policy_layer_sparsity.csv', index=False)
m0_mean = float(msum.loc[msum.cell == 'M0', 'mean'].iloc[0])
table = []
for policy in POLICIES:
    table.append({'policy': policy, 'mean_macro_f1': float(msum.loc[msum.cell == policy, 'mean'].iloc[0]),
                  'mean_macro_f1_loss': m0_mean - float(msum.loc[msum.cell == policy, 'mean'].iloc[0]),
                  'sd_macro_f1': float(msum.loc[msum.cell == policy, 'sd'].iloc[0]),
                  'classes_affected_ge3of5': int((summ[summ.cell == policy].affected_frequency >= 0.6).sum()),
                  'conv0_sparsity': float(lsdf[lsdf.cell == policy]['conv.0'].mean()), 'conv3_sparsity': float(lsdf[lsdf.cell == policy]['conv.3'].mean()),
                  'head_sparsity': float(lsdf[lsdf.cell == policy]['head'].mean()), 'prunable_sparsity': float(lsdf[lsdf.cell == policy]['prunable_sparsity'].mean())})
table = pd.DataFrame(table); table.to_csv(OUT / 'cnn_policy_comparison.csv', index=False)
print(f'M0 five-seed macro-F1: {m0_mean:.4f}\n'); print(table.round(4).to_string(index=False))
print('\nmost-affected classes under each policy:')
for policy in POLICIES:
    s = summ[summ.cell == policy].sort_values('mean_recall_loss', ascending=False).head(6)
    print(f'  {policy}: ' + ', '.join(f'{r["class"]} {r.mean_recall_loss:.2f}' for _, r in s.iterrows()))

In [ ]:
# Mechanism on the two new policies (anchor seed): probes on the archived layer-wise collapsed set, plus head refit
seed = ANCHOR
m0, le, scaler, _ = load_anchor(DATASET, ARCH, 'M0_paired', seed, arch_kwargs=ARCH_KW)
ref_collapsed = list(summ[(summ.cell == 'layerwise80_all') & (summ.affected_frequency >= 0.6)]['class'])
print(f'{len(ref_collapsed)} classes collapsed under the archived layer-wise policy (>=3/5 seeds)')
mech_rows = []
for policy in ['layerwise80_protect_conv0', 'global80']:
    matched = policy; pm = models_out[(seed, policy)]; collapsed = ref_collapsed
    pdf = pd.DataFrame(columns=['label', 'auc_M0', 'auc_compressed', 'auc_drop'])
    if collapsed:
        fit = {}
        for which in ('train', 'val'):
            f0, _, y = EXP.extract_features(m0, df, splits, scaler, feat_cols, le, which=which)
            fc, _, _ = EXP.extract_features(pm, df, splits, scaler, feat_cols, le, which=which); fit[which] = (f0, fc, y)
        f0_te, _, y_te = EXP.extract_features(m0, df, splits, scaler, feat_cols, le, which='test')
        fc_te, _, _ = EXP.extract_features(pm, df, splits, scaler, feat_cols, le, which='test')
        f0_fit = np.concatenate([fit['train'][0], fit['val'][0]]); fc_fit = np.concatenate([fit['train'][1], fit['val'][1]]); y_fit = np.concatenate([fit['train'][2], fit['val'][2]])
        rng = np.random.default_rng(seed); keep = []
        for c in np.unique(y_fit):
            idx = np.where(y_fit == c)[0]; keep.append(rng.choice(idx, 8000, replace=False) if len(idx) > 8000 else idx)
        keep = np.concatenate(keep); f0_fit, fc_fit, y_fit = f0_fit[keep], fc_fit[keep], y_fit[keep]
        def probe(f_fit, f_te, c):
            yb, yt_ = (y_fit == c).astype(int), (y_te == c).astype(int)
            if yb.sum() < 5 or yt_.sum() < 2: return np.nan
            return roc_auc_score(yt_, LogisticRegression(max_iter=500, C=1.0, class_weight='balanced').fit(f_fit, yb).predict_proba(f_te)[:, 1])
        prow = []
        for cname in collapsed:
            c = int(np.where(le.classes_ == cname)[0][0]); a0, ac = probe(f0_fit, f0_te, c), probe(fc_fit, fc_te, c)
            prow.append({'label': cname, 'auc_M0': a0, 'auc_compressed': ac, 'auc_drop': a0 - ac})
        pdf = pd.DataFrame(prow); print(pdf.round(4).to_string(index=False))
    pdf.to_csv(OUT / f'cnn_{matched}_leakage_safe_probe.csv', index=False)

    Lval, yval, Ltest, ytest = mitigate.refit_head(pm, df, splits, scaler, feat_cols, le, epochs=15, lr=1e-2, batch_size=4096, seed=seed)
    refit_f1 = f1_score(ytest, np.asarray(Ltest).argmax(1), average='macro')
    cal = calibration_summary(torch.softmax(torch.tensor(Ltest), dim=1).numpy(), ytest); cal.insert(0, 'cell', f'cnn_{matched}_head_refit')
    cal.to_csv(OUT / f'cnn_{matched}_head_refit_calibration.csv', index=False)
    print(f'\nhead refit macro-F1 under {matched}: {refit_f1:.4f}')
    mech_rows.append({'policy': policy, 'min_probe_auc_on_ref_collapsed': float(pdf.auc_compressed.min()) if len(pdf) else np.nan, 'head_refit_macro_f1': refit_f1})
mech_df = pd.DataFrame(mech_rows); mech_df.to_csv(OUT / 'cnn_policy_mechanism.csv', index=False); print(mech_df.round(4).to_string(index=False))

In [ ]:
# Gate verdict - written whichever way it falls
A = table[table.policy == 'layerwise80_all'].iloc[0]; B = table[table.policy == 'layerwise80_protect_conv0'].iloc[0]; C = table[table.policy == 'global80'].iloc[0]
ref_collapses = bool(A.mean_macro_f1_loss > 0.15 and A.classes_affected_ge3of5 >= 4)
b_eliminates = bool(B.mean_macro_f1_loss < 0.10 and B.classes_affected_ge3of5 < 4)
c_eliminates = bool(C.mean_macro_f1_loss < 0.10 and C.classes_affected_ge3of5 < 4)
verdict = pd.DataFrame([
 {'criterion': 'A_reference_reproduces_collapse', 'value': f'loss {A.mean_macro_f1_loss:.3f}, {int(A.classes_affected_ge3of5)} classes', 'pass': ref_collapses},
 {'criterion': 'B_protect_conv0_eliminates_collapse', 'value': f'loss {B.mean_macro_f1_loss:.3f}, {int(B.classes_affected_ge3of5)} classes, conv0 sparsity {B.conv0_sparsity:.2f}', 'pass': b_eliminates},
 {'criterion': 'C_global80_eliminates_collapse', 'value': f'loss {C.mean_macro_f1_loss:.3f}, {int(C.classes_affected_ge3of5)} classes, conv0 sparsity {C.conv0_sparsity:.2f}', 'pass': c_eliminates},
])
print(verdict.to_string(index=False))
h_input = bool(ref_collapses and b_eliminates)
print('\nH-input (input-interface starvation from layer-wise pruning of conv.0) supported:', h_input)
if ref_collapses and not b_eliminates: print('conv.0 protection does NOT remove the collapse: the cause lies deeper in the convolutional body (kernel-size ablation next).')
verdict.to_csv(OUT / 'cnn_policy_gate_verdict.csv', index=False)
write_json(OUT / 'cnn_policy_environment.json', {'arch_kwargs': ARCH_KW, 'policies': POLICIES, 'seeds': SEEDS, 'environment': environment_record()})

In [ ]:
# --- Commit + push: main only, this notebook's own files only ---
import subprocess, shutil, glob
_b = subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert _b == 'main', f'checked-out branch is {_b!r}; run `git checkout main` first'
subprocess.run(['git', 'config', '--global', 'user.name', 'Md Anas Biswas'], check=True)
subprocess.run(['git', 'config', '--global', 'user.email', 'anasbiswas@gmail.com'], check=True)
cred = '/content/drive/MyDrive/IoT_Trust_Research/.git-credentials'
if os.path.exists(cred):
    shutil.copy(cred, '/root/.git-credentials'); subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
_own = 'notebooks/34_cnn_pruning_policy_input_interface.ipynb'
if os.path.exists(_own):
    d = _json.load(open(_own))
    for c in d.get('cells', []):
        if c.get('cell_type') == 'code': c['outputs'] = []; c['execution_count'] = None
    _json.dump(d, open(_own, 'w'), indent=1)
subprocess.run(['git', 'add', _own] + glob.glob('results/tables/comnet/cnn_policy_*') + glob.glob('results/tables/comnet/cnn_layerwise80_protect_conv0_*') + glob.glob('results/tables/comnet/cnn_global80_*'), check=True)
r = subprocess.run(['git', 'commit', '-m', 'notebook 34: CNN pruning-policy test - conv.0 protection and global magnitude vs archived layer-wise on five paired baselines; input-interface hypothesis gate'], capture_output=True, text=True)
print(r.stdout or r.stderr)
print(subprocess.run(['git', 'push'], capture_output=True, text=True).stderr or 'pushed')
print(subprocess.run(['git', 'log', '--oneline', '-2'], capture_output=True, text=True).stdout)